In [ ]:

import os, sys, glob, json, time, subprocess
T0 = time.time(); TRAN_GIO = 11.3; PHI4_CROPS = 16
subprocess.run("pip uninstall -y -q torchao; pip install -q transformers==4.48.2 peft==0.13.2 accelerate==1.3.0 backoff soundfile scipy", shell=True)
from huggingface_hub import snapshot_download
snapshot_download("microsoft/Phi-4-multimodal-instruct")
def tim(mau):
    r = sorted(glob.glob(f"/kaggle/input/**/{mau}", recursive=True)); assert r, mau; return r[0]
SOM  = os.path.dirname(tim("som/som.jsonl"))
CODE = os.path.dirname(tim("som_listener.py"))
assert "14/9-b" in open(f"{CODE}/som_listener.py").read(), "mã cũ"
IMG  = os.path.dirname(os.path.dirname(tim("test_ac/images/ep18171_s1.png")))
REF  = tim("ref/chon_phi4_s1_101.jsonl")
OUT  = "/kaggle/working/som"; os.makedirs(OUT, exist_ok=True)
TOM  = open(f"{OUT}/tom_tat.txt", "a")
def ghi(*a):
    s = f"[{(time.time()-T0)/3600:5.2f} h] " + " ".join(map(str, a)); print(s, flush=True); TOM.write(s + "\n"); TOM.flush()
ghi("SOM", SOM, "CODE", CODE, "IMG", IMG)
def chay(ten, cau, gpu, extra=()):
    log = open(f"{OUT}/{ten}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1",
           "HF_HUB_DISABLE_PROGRESS_BARS": "1", "SOM_PHI4_CROPS": str(PHI4_CROPS), "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"}
    cmd = [sys.executable, f"{CODE}/som_listener.py", "--backend", "phi4", "--img-root", IMG, "--som", f"{SOM}/som.jsonl",
           "--cau", f"{SOM}/cau_{cau}.jsonl", "--out", f"{OUT}/{ten}.jsonl", *map(str, extra)]
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
def cho(ds, nhip=120):
    while any(p.poll() is None for _, p in ds):
        time.sleep(nhip)
        if (time.time() - T0) / 3600 > TRAN_GIO:
            ghi("chạm trần giờ — dừng"); [p.terminate() for _, p in ds]; time.sleep(20); break
        for ten, _ in ds:
            d = open(f"{OUT}/{ten}.log").read().strip().splitlines()[-1:] or [""]
            print(f"  {(time.time()-T0)/3600:5.2f} h · {ten}: {d[0][:140]}", flush=True)
    for ten, p in ds: ghi(ten, "mã thoát", p.poll())
def doc(f):
    D = {}
    for l in open(f):
        try: o = json.loads(l); D[(o["episode_id"], o["step_id"])] = o
        except json.JSONDecodeError: pass
    return D

# 0. phép kiểm tái lập: chạy lại S1/101 trên lát 200, so với kết quả đã có ở runs/som (cùng mã, cùng cấu hình)
cho([("kiem_s1_101_mau200", chay("kiem_s1_101_mau200", "s1_101", "0", ["--mau", 200]))])
cu, moi = doc(REF), doc(f"{OUT}/kiem_s1_101_mau200.jsonl")
ch = [k for k in moi if k in cu]
lech = sum(moi[k]["chon"] != cu[k]["chon"] for k in ch)
ghi(f"KIỂM TÁI LẬP S1/101: {len(ch)} bước chung · lệch lựa chọn {lech} (cần ~0; lệch nhiều = môi trường khác bản đã chấm)")

# 1. chấm đủ 4.463 bước: ck500 rồi r3, chia 2 GPU
for nh in ["s1_202_diff"]:
    if TRAN_GIO - (time.time() - T0) / 3600 < 2.5: ghi("bỏ qua", nh, ": không đủ giờ"); break
    ten = f"chon_phi4_{nh}"
    cho([(f"{ten}_s{i}", chay(f"{ten}_s{i}", nh, str(i), ["--shard", f"{i}/2"])) for i in (0, 1)])
    with open(f"{OUT}/{ten}.jsonl", "w") as g:
        for i in (0, 1):
            if os.path.exists(f"{OUT}/{ten}_s{i}.jsonl"): g.write(open(f"{OUT}/{ten}_s{i}.jsonl").read())
    D = doc(f"{OUT}/{ten}.jsonl")
    ghi(f"{ten}: {len(D)} bước · chọn đúng {100*sum(o['dung'] for o in D.values())/max(1,len(D)):.2f}%  {'(ĐỦ)' if len(D)==4463 else '(CHƯA ĐỦ)'}")
subprocess.run(f"cd /kaggle/working && zip -qr som_s1_202.zip som", shell=True)
ghi("KẾT THÚC")
